# Práctica supervisada · Regresión lineal simple

**Recorrido:** SQL + Python  
**Tiempo orientativo del notebook:** 12–15 minutos  
**Carácter:** obligatoria.

> Realizá solamente uno de los dos recorridos equivalentes: **SQL + Python** o **SQL + R**.

En esta práctica vas a aplicar lo trabajado en la Actividad 1 a un caso nuevo. Analizaremos la relación entre el **tamaño de un dataset** y el **tiempo requerido para procesarlo**.

El objetivo no es repetir mecánicamente código, sino interpretar:

- la relación entre las variables;
- la pendiente del modelo;
- una predicción;
- un residuo;
- el valor de \(R^2\);
- una limitación de la estimación.

Al finalizar el notebook, completá la **Práctica supervisada en Moodle**. Esa actividad dejará registrada una evidencia breve de tu trabajo para el seguimiento del equipo de tutoría.

## 1. Preparar los datos

Ejecutá la celda siguiente. El notebook genera internamente un conjunto de mediciones de procesamiento y crea una base SQLite en memoria.

In [ ]:
import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

n = np.arange(1, 41)

filas_miles = np.round(5 + 2.4 * n + 1.8 * np.sin(n * 0.55), 1)
ruido = 0.9 * np.sin(n * 1.31) + 0.45 * np.cos(n * 0.37)
tiempo_seg = np.round(1.6 + 0.115 * filas_miles + ruido, 2)

procesamiento = pd.DataFrame({
    "medicion": n,
    "filas_miles": filas_miles,
    "tiempo_seg": tiempo_seg
})

con = sqlite3.connect(":memory:")
procesamiento.to_sql("procesamiento", con, index=False, if_exists="replace")

print(f"Datos preparados: {len(procesamiento)} mediciones.")

## 2. Recuperar con SQL las variables del análisis

Queremos estimar `tiempo_seg` a partir de `filas_miles`.

In [ ]:
consulta = """
SELECT
    filas_miles,
    tiempo_seg
FROM procesamiento
ORDER BY filas_miles
"""
df = pd.read_sql_query(consulta, con)
df.head(8)

## 3. Observar la relación

Antes de ajustar el modelo, observá el diagrama de dispersión.

In [ ]:
plt.figure(figsize=(7, 4.5))
plt.scatter(df["filas_miles"], df["tiempo_seg"], alpha=0.75)
plt.xlabel("Tamaño del dataset (miles de filas)")
plt.ylabel("Tiempo de procesamiento (segundos)")
plt.title("Tamaño del dataset y tiempo de procesamiento")
plt.grid(alpha=0.2)
plt.show()

**Pensá antes de continuar:** ¿la relación parece aproximadamente lineal?, ¿es positiva o negativa?

## 4. Ajustar la regresión lineal simple

In [ ]:
X = df[["filas_miles"]]
y = df["tiempo_seg"]

modelo = LinearRegression()
modelo.fit(X, y)

pendiente = modelo.coef_[0]
intercepto = modelo.intercept_

print(f"Pendiente:  {pendiente:.4f}")
print(f"Intercepto: {intercepto:.4f}")

### Interpretación guiada

La **pendiente** indica cuánto aumenta, en promedio, el tiempo estimado por el modelo cuando el dataset crece en **1.000 filas**.

Completá mentalmente esta frase antes de seguir:

> “Por cada 1.000 filas adicionales, el modelo estima que el tiempo de procesamiento aumenta aproximadamente ______ segundos.”

## 5. Generar una predicción

Estimemos el tiempo de procesamiento para un dataset de **75.000 filas**.

In [ ]:
nuevo = pd.DataFrame({"filas_miles": [75]})
pred_75 = modelo.predict(nuevo)[0]

print(f"Tiempo estimado para 75.000 filas: {pred_75:.2f} segundos")

Recordá que esta predicción tiene sentido porque 75.000 filas se encuentra dentro del rango aproximado de valores observados. Usar la misma recta muy por fuera de ese rango sería **extrapolar**.

## 6. Analizar un residuo

Tomemos una de las observaciones y comparemos su valor real con el valor predicho.

In [ ]:
df["predicho"] = modelo.predict(X)
df["residuo"] = df["tiempo_seg"] - df["predicho"]

caso = df.iloc[20]
caso

Interpretá el signo del residuo:

- si es **positivo**, el valor real fue mayor que el estimado;
- si es **negativo**, el modelo estimó un valor mayor que el observado.

## 7. Analizar \(R^2\)

In [ ]:
r2 = r2_score(y, df["predicho"])
print(f"R² = {r2:.3f}")

En este caso, \(R^2\) indica qué proporción de la variabilidad observada en el tiempo de procesamiento queda representada por la relación lineal con el tamaño del dataset.

No lo interpretes como una garantía absoluta de calidad: también deben considerarse el contexto, el rango de datos y otros factores que podrían influir sobre el tiempo de procesamiento.

## 8. Cierre

Antes de pasar al cuestionario supervisado, asegurate de poder responder:

1. ¿Cuál es la variable predictora y cuál es la variable objetivo?
2. ¿Qué significa la pendiente obtenida?
3. ¿Qué valor estimó el modelo para 75.000 filas?
4. ¿Qué significa el signo del residuo observado?
5. ¿Qué información aporta \(R^2\)?
6. ¿Por qué no conviene extrapolar sin criterio?

Ahora completá la **Práctica supervisada · Regresión lineal simple** en Moodle.

In [ ]:
con.close()